# Reducción de Dimensionalidad mediante Embeddings (t-SNE / Manifold Learning)

Mientras que PCA asume relaciones lineales entre las variables, técnicas de *Manifold Learning* como **t-SNE** (t-Distributed Stochastic Neighbor Embedding) construyen **embeddings** no lineales. Esto permite proyectar las cientos de características tácticas de la Fórmula 1 en un espacio de baja dimensionalidad (2D o 3D) preservando la estructura topológica local (eventos tácticos similares se agrupan juntos).

Este notebook explora la creación de estos embeddings para descubrir clústeres ocultos que PCA podría haber pasado por alto debido a su naturaleza lineal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style="darkgrid")
plt.rcParams['figure.dpi'] = 120

## 1. Carga y Preparación de Datos Raw

In [ ]:
# Cargar el dataset crudo de eventos tácticos
df_raw = pd.read_parquet('../data/features/tactical_events_v3.parquet')

# Seleccionar solo variables numéricas para el embedding
numeric_cols = df_raw.select_dtypes(include=['float64', 'float32', 'int64', 'int32']).columns
X_raw = df_raw[numeric_cols].copy()

# Eliminar IDs y variables target que no deben influenciar el embedding
cols_to_drop = [col for col in X_raw.columns if 'id' in col.lower() or 'pos_change' in col.lower()]
X_features = X_raw.drop(columns=cols_to_drop, errors='ignore')

# Imputación y Escalado
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Dataset preparado para embeddings: {X_scaled.shape[0]} eventos x {X_scaled.shape[1]} features.")

## 2. Generación de Embeddings 2D y 3D

**Justificación de Dimensionalidad:** Proyectamos el espacio latente estrictamente en **2 y 3 dimensiones** por dos motivos técnicos críticos:
1. **Limitación Algorítmica (Barnes-Hut):** La implementación óptima de t-SNE usa la aproximación *Barnes-Hut* para reducir la complejidad computacional. Matemáticamente, este algoritmo está diseñado para soportar únicamente un máximo de 3 dimensiones. Usar 4 o más requeriría el método 'exact', haciendo el pipeline inviable computacionalmente para esta volumetría.
2. **Interpretabilidad Visual:** A diferencia de PCA (del cual extrajimos 15 dimensiones para Machine Learning puro), el Manifold Learning se usa para el descubrimiento y validación topológica humana. Tres componentes es el límite máximo que un analista puede visualizar, rotar e interpretar gráficamente de forma efectiva.

In [ ]:
# Embedding 2D usando t-SNE
# Perplexity controla el balance entre atención a aspectos locales vs globales (típicamente entre 5 y 50)
tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
embeddings_2d = tsne_2d.fit_transform(X_scaled)

# Guardamos los embeddings en el dataframe original para visualización
df_raw['Emb_2D_X'] = embeddings_2d[:, 0]
df_raw['Emb_2D_Y'] = embeddings_2d[:, 1]

In [ ]:
# Embedding 3D usando t-SNE para explorar profundidad geométrica
tsne_3d = TSNE(n_components=3, perplexity=30, random_state=42, init='pca', learning_rate='auto')
embeddings_3d = tsne_3d.fit_transform(X_scaled)

df_raw['Emb_3D_X'] = embeddings_3d[:, 0]
df_raw['Emb_3D_Y'] = embeddings_3d[:, 1]
df_raw['Emb_3D_Z'] = embeddings_3d[:, 2]

## 3. Análisis Visual del Espacio de Embeddings
En un espacio t-SNE, las distancias relativas importan localmente: si dos puntos están cerca, representan batallas con dinámicas de telemetría y desgaste sumamente parecidas.

### 3.1. Proyección 2D: Análisis de Tipos de Eventos y Éxito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Forzamos pos_change a numérico por seguridad
if 'pos_change' in df_raw.columns:
    df_raw['pos_change_num'] = pd.to_numeric(df_raw['pos_change'], errors='coerce')
    df_raw['Success'] = np.where(df_raw['pos_change_num'] > 0, 'Adelantamiento Exitoso', 'Intento Fallido')

# Gráfico A: Separación por Tipo de Evento (Attacks vs Undercuts)
if 'event_type' in df_raw.columns:
    sns.scatterplot(
        x='Emb_2D_X', y='Emb_2D_Y', 
        hue='event_type', 
        data=df_raw, 
        palette='Set1', 
        alpha=0.7, 
        s=40, 
        ax=axes[0]
    )
    axes[0].set_title('Embeddings 2D: Tipo de Táctica (Ataque vs Undercut)', fontsize=13)

# Gráfico B: Separación por Éxito del Evento
if 'Success' in df_raw.columns:
    sns.scatterplot(
        x='Emb_2D_X', y='Emb_2D_Y', 
        hue='Success', 
        data=df_raw, 
        palette={'Adelantamiento Exitoso': '#2ecc71', 'Intento Fallido': '#e74c3c'}, 
        alpha=0.6, 
        s=40, 
        ax=axes[1]
    )
    axes[1].set_title('Embeddings 2D: Tasa de Éxito de la Maniobra', fontsize=13)

plt.tight_layout()
plt.show()

### 3.2. Proyección de Densidad Topológica (Hexbin)
Para identificar los "centros de gravedad" o las tácticas más comunes.

In [ ]:
plt.figure(figsize=(10, 8))
hb = plt.hexbin(df_raw['Emb_2D_X'], df_raw['Emb_2D_Y'], gridsize=40, cmap='inferno', mincnt=1)
cb = plt.colorbar(hb)
cb.set_label('Concentración de Eventos')
plt.title('Mapa de Densidad del Espacio Táctico (Hexbin)', fontsize=15)
plt.xlabel('Dimensión Latente 1')
plt.ylabel('Dimensión Latente 2')
plt.show()

### 3.3. Overlay de Features Físicos Continuos
Coloreamos el embedding en base a una variable contínua de la telemetría para comprobar qué zonas del mapa latente representan velocidades altas o gran degradación.

In [ ]:
feature_to_paint = 'att_st_speed_mean'

if feature_to_paint in df_raw.columns:
    # Convertimos a numérico en caso de ser necesario
    df_raw[feature_to_paint] = pd.to_numeric(df_raw[feature_to_paint], errors='coerce')
    
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(
        df_raw['Emb_2D_X'], df_raw['Emb_2D_Y'], 
        c=df_raw[feature_to_paint], 
        cmap='coolwarm', 
        alpha=0.8, 
        s=30
    )
    plt.colorbar(scatter, label='Velocidad Promedio en Recta (km/h)')
    plt.title(f'Overlay de Feature: {feature_to_paint} sobre Embeddings', fontsize=14)
    plt.show()

### 3.4. Exploración 3D del Espacio Latente
Un embedding 3D reduce el apelotonamiento (crowding problem) permitiendo desenredar grupos de datos complejos.

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

if 'event_type' in df_raw.columns:
    unique_events = df_raw['event_type'].dropna().unique()
    colors = sns.color_palette('bright', len(unique_events))
    
    for idx, event in enumerate(unique_events):
        subset = df_raw[df_raw['event_type'] == event]
        ax.scatter(
            subset['Emb_3D_X'], subset['Emb_3D_Y'], subset['Emb_3D_Z'],
            label=event, color=colors[idx], alpha=0.6, s=25
        )

    ax.set_xlabel('Dim Latente 1')
    ax.set_ylabel('Dim Latente 2')
    ax.set_zlabel('Dim Latente 3')
    ax.set_title('Topología 3D de Eventos Tácticos (t-SNE Embeddings)', fontsize=15)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 4. Extracción y Guardado del Dataset Enriquecido
Guardamos el dataframe original ahora potenciado con las coordenadas latentes 2D y 3D, lo cual será sumamente útil si se desean aplicar modelos de Deep Learning o clasificadores de gradient boosting (XGBoost) sobre estas coordenadas.

In [ ]:
cols_to_save = ['event_id', 'Emb_2D_X', 'Emb_2D_Y', 'Emb_3D_X', 'Emb_3D_Y', 'Emb_3D_Z']
available_cols = [c for c in cols_to_save if c in df_raw.columns]

df_embeddings = df_raw[available_cols]
df_embeddings.to_parquet('../data/features/tactical_embeddings.parquet', index=False)
print(f"Embeddings guardados exitosamente en 'tactical_embeddings.parquet'. Dimensiones: {df_embeddings.shape}")